# S6E9: the 0.9465 "rule recipe", decoded — what moves the needle and what doesn't

The public LB of Playground S6E9 is saturated around **0.9465**. Sitting on top of the
model-blend plateau is a small **rule recipe** (widely discussed by the community, e.g.
the published blends anchored on jazivxt's `submission_latest_best.csv`): four hard rank
shifts applied on top of a model's rank scores.

This notebook:
1. shows the **data-side evidence** behind the big marginal signals (subsidy, income
   bands, commute),
2. decodes the **mechanics** of the recipe (why "+10" is a *hard block*, not a nudge),
3. reports an honest **negative result**: we took 10 rule-delta variants on top of the
   recipe output to the LB (10 submissions, Sep 19) — **every new rule we mined from
   train made the score worse**, and the dose–response is monotone.

If you are plateaued at ~0.9464, this may save you a few submissions.

## 1. Setup

Competition data attached; everything runs offline on CPU.

In [ ]:
import numpy as np, pandas as pd, glob, os
import matplotlib.pyplot as plt

# locate the competition files regardless of mount name
cands = [p for p in glob.glob('/kaggle/input/**/train.csv', recursive=True)
         if os.path.exists(os.path.join(os.path.dirname(p), 'test.csv'))]
assert cands, 'competition data not mounted'
IN = os.path.dirname(cands[0])
print('data dir:', IN)
tr = pd.read_csv(f'{IN}/train.csv')
te = pd.read_csv(f'{IN}/test.csv')
y = (tr['Will_Buy_EV'].astype(str).str.strip().str.lower() == 'yes').astype(float).to_numpy()
g = y.mean()
print(f"train {len(tr):,} rows | test {len(te):,} rows | global buy rate {g:.5f}")

## 2. The big marginal signals

Three features dominate the margins. The most striking one is **subsidy**:
`Subsidy_Available == 'No'` collapses the buy rate to ~0.6% (global mean 17.5%).

In [ ]:
def band_table(x, step, name, min_n=400, min_dev=0.02):
    lo = np.floor(x / step).astype(int)
    df = pd.DataFrame({'bin': lo, 'y': y})
    agg = df.groupby('bin')['y'].agg(['mean', 'count'])
    agg['dev'] = agg['mean'] - g
    agg['lo'] = agg.index * step; agg['hi'] = (agg.index + 1) * step
    sel = agg[(agg['count'] >= min_n) & (agg['dev'].abs() >= min_dev)]
    print(f"--- {name} ---")
    print(sel[['lo','hi','count','dev']].to_string(index=False))
    return sel

inc = tr['Annual_Income_USD'].to_numpy(float)
km  = tr['Daily_Commute_km'].to_numpy(float)
sub = tr['Subsidy_Available'].astype(str).to_numpy()

_ = band_table(inc, 20000, 'income $20k')
_ = band_table(km, 10, 'commute 10km')

In [ ]:
df = pd.DataFrame({'sub': sub, 'y': y})
print(df.groupby('sub')['y'].agg(['mean','count']).to_string())

**But** — and this is the point of this notebook — marginal signal ≠ LB gain.
The models we all train already encode these features. Post-hoc rules only pay where
the *model ranking* systematically fails, and that territory is small.

## 3. The recipe's mechanics: rules are hard *blocks*

The public recipe is:

```
final = lexrank( rank01(anchor) + 0.05 * rank01(lasso) + shift , lasso )
```

with four shifts on the anchor's rank scores:

| rule | shift |
|---|---|
| `income >= 170537` | **+10** |
| `income in [31004, 41970]` | **−10** |
| `commute >= 83` | **−5** |
| `income == 30000 & no subsidy & (env==1 | anxiety in {M,H})` | **−5** |

Since `rank01(...) + 0.05*rank01(lasso)` lives in `[0, 1.05]`, a shift of **±10 is not a
nudge — it is a hard block** that carries its rows to the very top/bottom of the ranking.
Within a block, ordering is inherited from the model score (ties broken by the lasso
rank). That's why the recipe works *despite* looking crude: it is a rank-domain
surgical tool, not a reweighting.

In [ ]:
# Toy baseline to demonstrate block semantics (no model tuning intended here)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

feats = ['Age','Annual_Income_USD','Daily_Commute_km','Number_of_Cars_Owned',
         'Charging_Stations_Near_Home','Charging_Stations_Near_Work',
         'Environmental_Concern_Level']
cats = ['Gender','City_Type','Current_Car_Type','Home_Charging_Possible',
        'Subsidy_Available','Range_Anxiety_Level']
def encode(d):
    z = pd.get_dummies(d[feats + cats], drop_first=True)
    return z.fillna(z.median(numeric_only=True))
X = encode(tr); Xt = encode(te)[X.columns]
sc = StandardScaler().fit(X)
m = LogisticRegression(max_iter=2000).fit(sc.transform(X), y)
base = pd.Series(m.predict_proba(sc.transform(Xt))[:, 1])
print('toy LR baseline ready:', base.shape)

In [ ]:
# Demonstrate what a '+10' hard block does to the ranking
N = len(base)
r01 = base.rank(pct=True).to_numpy()
shift = np.where(te['Annual_Income_USD'].to_numpy(float) >= 170537, 10.0, 0.0)
final = r01 + shift
pos_before = pd.Series(r01).rank(method='first').to_numpy()
pos_after  = pd.Series(final).rank(method='first').to_numpy()
moved = te['Annual_Income_USD'].to_numpy(float) >= 170537
print(f"rows in the >=170537 block: {moved.sum():,} of {N:,}")
print(f"mean rank position before: {pos_before[moved].mean():,.0f}  ->  after: {pos_after[moved].mean():,.0f}")
print("=> the block jumps above every un-shifted row: a HARD BLOCK, not a nudge")

## 4. The honest negative result: 10 rule-delta probes, all worse

We took the recipe output (LB **0.94651**) and applied *additional* shifts mined from
train — as **graded** deltas (`|delta| < 1` ⇒ shifts by ~`delta*N` rank positions) and
**hard** deltas (`|delta| >= 1` ⇒ full blocks). Ten submissions, Sep 19:

| probe | rows | delta | LB | Δ vs 0.94651 |
|---|---|---|---|---|
| Subsidy=No & income<100k — soft | 80,579 | −0.05 | 0.94647 | −4e-5 |
| Subsidy=No & income<100k — hard | 80,579 | −1.0 | 0.94534 | −117e-5 |
| commute 65–83 — hard | 6,547 | −1.0 | 0.93783 | −868e-5 |
| commute 65–83 — soft | 6,547 | −0.05 | 0.94636 | −15e-5 |
| income 156k–170537 — hard | 1,220 | +1.0 | 0.94398 | −253e-5 |
| income 140k–170537 — graded | 7,866 | +0.30 | 0.94131 | −520e-5 |
| **the 4 existing rules — softened ×0.0005** | 3,731 | graded | **0.94651** | **0** |
| Subsidy=Yes & income<50k — soft | 19,768 | −0.05 | 0.94624 | −27e-5 |
| top-ext + commute-ext combo | 7,725 | ±1.0 | 0.93557 | −1094e-5 |
| RangeAnxiety=High — soft | 1,029 | −0.05 | 0.94650 | −1e-5 (noise) |

Three conclusions:

1. **Every newly mined rule hurt.** The models already rank these slices correctly;
   mining deviations from train double-dips the same data.
2. **Dose–response is monotone**: the more rows you move, the worse it gets
   (soft > hard within every pair).
3. **Hard vs graded is invisible** on the existing small slices (softened ×0.0005
   scored *identically* 0.94651) — the recipe's value is not block hardness; it is
   the *choice* of those four specific slices, presumably tuned on LB feedback by the
   recipe authors.

## 5. Takeaways if you're plateaued at ~0.9464

- The rule recipe family is **closed**: stacking more rules on top of it only burns
  submissions (see §4).
- Blend diversity on the same feature matrix buys ≤ ~4e-5 — consistent with what the
  GM notes going around suggest.
- The remaining headroom is in **genuinely different signal sources** (different target
  encoding, different feature matrices), not in post-hoc rank surgery.
- If you reproduce the recipe, keep the four rules *exactly as published* — we measured
  that their graded variants are LB-equivalent, so the block structure is robust, but
  the slices themselves are not negotiable.

*Reproducibility: every number in §4 comes from our own submissions on Sep 19, 2026
(team DefiAudit). The recipe itself is community-published; this notebook only decodes
its mechanics and stress-tests its boundary.*